# Compare trained closures

Select any number of trained closure directories below. All selected models must use the same grid and validation trajectories. This notebook reuses the rollout and diagnostic implementations in `scripts/compare_models.py` and `model/utils/diagnostics.py`.

In [1]:
import os
import sys
from pathlib import Path

WORKSPACE_ROOT = Path.cwd()
if not (WORKSPACE_ROOT / 'model').is_dir():
    WORKSPACE_ROOT = WORKSPACE_ROOT.parent
if not (WORKSPACE_ROOT / 'model').is_dir():
    raise FileNotFoundError('Could not locate the workspace root containing model/')
sys.path.insert(0, str(WORKSPACE_ROOT))

from scripts.compare_models import (
    build_config_from_metadata,
    load_model_from_closure_dir,
    load_validation_data,
    rollout_model,
)
from model.core.grid import build_grid
from model.utils.diagnostics import build_diagnostic

In [2]:
# Replace these examples with the closure directories you want to compare.
MODEL_DIRS = [
    WORKSPACE_ROOT / 'saved_closures' / 'resnet' / '512_to_32' / '01',
    WORKSPACE_ROOT / 'saved_closures' / 'mlp' / '512_to_32' / '01',
]
VALIDATION_DATA = WORKSPACE_ROOT / 'data' / 'data_hr512_nx32_01'
N_TRAJS = 5
OUTPUT_DIR = WORKSPACE_ROOT / 'outputs' / 'model_comparison'

# Available multi-model diagnostics:
# multi_model_comparison: KE spectra and spectral relative errors
# multi_model_loss: training and test loss histories
# pareto_validation: validation loss vs rollout survival time
DIAGNOSTICS = [
    'multi_model_comparison',
    'multi_model_loss',
    'pareto_validation',
]
CADENCE = 10

if not MODEL_DIRS:
    raise ValueError('Add at least one closure directory to MODEL_DIRS')
if len(set(DIAGNOSTICS)) != len(DIAGNOSTICS):
    raise ValueError('DIAGNOSTICS contains duplicates')

In [3]:
models = [load_model_from_closure_dir(path) for path in MODEL_DIRS]

# These comparison diagnostics use one shared grid and truth dataset.
grid_keys = ('nx', 'ny', 'hr_nx', 'nz')
reference_params = models[0]['metadata']['parameters']
for model_info in models[1:]:
    params = model_info['metadata']['parameters']
    mismatches = {
        key: (reference_params.get(key), params.get(key))
        for key in grid_keys
        if reference_params.get(key) != params.get(key)
    }
    if mismatches:
        raise ValueError(
            f"Incompatible grid for {model_info['model_name']}: {mismatches}"
        )

validation_trajs = load_validation_data(VALIDATION_DATA)
if N_TRAJS < 1:
    raise ValueError('N_TRAJS must be at least 1')
n_trajs = min(N_TRAJS, len(validation_trajs))
if n_trajs < N_TRAJS:
    print(f'Using {n_trajs} available trajectories (requested {N_TRAJS}).')

results = {}
for model_info in models:
    name = model_info['model_name']
    rollout = rollout_model(model_info, validation_trajs, n_trajs=n_trajs)
    results[name] = {
        **rollout,
        'loss_history': model_info['loss_history'],
        'metadata': model_info['metadata'],
    }

Loading model from: c:\Users\C5044892\Documents\Code\QG\saved_closures\resnet\512_to_32\01


TypeError: build_closure() got an unexpected keyword argument 'model_type'

In [ ]:
cfg = build_config_from_metadata(models[0]['metadata'])
trajectories = {
    'truth': validation_trajs[:n_trajs],
    'grid': build_grid(params=cfg['params']),
    'model_names': list(results),
    'model_results': results,
}
for name, result in results.items():
    trajectories[f'pred_{name}'] = result['pred']
    trajectories[f'loss_history_{name}'] = result['loss_history']

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for diagnostic_name in DIAGNOSTICS:
    diagnostic = build_diagnostic(diagnostic_name)
    output_path = OUTPUT_DIR / f'{diagnostic_name}.{diagnostic.output}'
    diagnostic.run(trajectories, str(output_path), CADENCE)
    print(f'Saved {output_path}')

In [ ]:
# Compact numeric summary from the shared rollouts.
import pandas as pd

comparison_summary = pd.DataFrame.from_dict(
    {
        name: {
            'MSE': result['mse_loss'],
            'MAE': result['mae_loss'],
            'Average max CFL': result['avg_cfl'],
        }
        for name, result in results.items()
    },
    orient='index',
)
comparison_summary